# 01 · Data acquisition and provenance
Every dataset is registered by its SHA-256 in the signed ledger the moment it enters. Raw HTTP responses are cached under `data/raw/cache`, so a second run is fully offline.

In [1]:
import json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from synera_core.config import find_repo_root, get_settings
warnings.filterwarnings("ignore")
ROOT = find_repo_root()
C = dict(bg="#061424", panel="#0b2037", cyan="#16B5FF", green="#6FD65B", amber="#FFB547", red="#FF4D5E", teal="#19E3B1", text="#EAF2FB", muted="#8FA6BF")
plt.rcParams.update({"figure.facecolor": C["bg"], "axes.facecolor": C["bg"], "axes.edgecolor": C["muted"], "axes.labelcolor": C["text"],
    "text.color": C["text"], "xtick.color": C["muted"], "ytick.color": C["muted"], "grid.color": "#1c3a5c", "axes.grid": True, "grid.linewidth": 0.5,
    "axes.prop_cycle": plt.cycler(color=[C["cyan"], C["green"], C["amber"], C["red"], C["teal"]]), "figure.dpi": 110, "font.size": 9})
SEED = get_settings().seeds.global_seed
RES = ROOT / "artifacts" / "results"

In [2]:
from synera_core.pipeline.registry import get_registry, summarize
from synera_core.provenance.store import open_ledger
ledger = open_ledger()
rows = []
for s in get_registry().values():
    sm = summarize(s)
    hit = ledger.find_by_artifact(sm["sha256"]) if sm.get("available") else []
    rows.append({"id": s.id, "track": s.track, "provenance": s.provenance, "kind": s.kind, "sha256": (sm.get("sha256") or "")[:16],
                 "ledger_block": hit[0][0].index if hit else None, "rows/samples": sm.get("rows", sm.get("samples"))})
pd.DataFrame(rows)

,id,track,provenance,kind,sha256,ledger_block,rows/samples
0,env_panel,B,REAL,covariates,638456eaf9bb9d9c,0,4144
1,benchmark_track_b,B,SIMULATED,panel,dc0e90b589265950,15,4122
2,osd-211,A,REAL,factorial,0a7c079900e84c44,3,21
3,osd-237,A,REAL,factorial,11c1ff659dcb91b8,8,21


In [3]:
cache = list((ROOT / "data" / "raw" / "cache").glob("*.raw"))
print(f"{len(cache)} cached raw API responses ({sum(p.stat().st_size for p in cache)/1e6:.1f} MB): re-running is offline")
v = ledger.verify(); print("ledger:", v["height"], "blocks, valid:", v["valid"])
pd.DataFrame([{"index": b.index, "type": b.block_type, "time": b.timestamp[:19], "artifacts": len(b.payload.get("artifact_hashes", []))}
              for b in ledger.iter_blocks()]).tail(8)

345 cached raw API responses (105.1 MB): re-running is offline
ledger: 23 blocks, valid: True


,index,type,time,artifacts
15,15,DATASET_REGISTERED,2026-10-14T14:01:16,1
16,16,DATASET_REGISTERED,2026-10-14T14:01:16,1
17,17,EXPERIMENT_RUN,2026-10-14T14:01:22,3
18,18,EXPERIMENT_RUN,2026-10-14T14:06:13,3
19,19,EXPERIMENT_RUN,2026-10-14T14:06:36,6
20,20,FINDING_VALIDATED,2026-10-14T14:06:55,1
21,21,FINDING_VALIDATED,2026-10-14T14:06:55,1
22,22,FINDING_VALIDATED,2026-10-14T14:06:55,1


The environmental panel combines Open-Meteo weather and air-quality archives for eight cities. Missing values are never filled; rows with missing stressors are dropped explicitly at analysis time.

In [4]:
env = pd.read_parquet(ROOT / "data/processed/env_panel.parquet")
display(env.groupby("city").agg(days=("date","count"), start=("date","min"), end=("date","max")))
print("missing share per column:"); display((env[["pm25","pm10","o3","no2","uv"]].isna().mean()*100).round(2).rename("% missing"))

,days,start,end
city,,,
Beijing,518,2022-08-01,2023-12-31
Cairo,518,2022-08-01,2023-12-31
Delhi,518,2022-08-01,2023-12-31
Jakarta,518,2022-08-01,2023-12-31
Los Angeles,518,2022-08-01,2023-12-31
Ludhiana,518,2022-08-01,2023-12-31
Mumbai,518,2022-08-01,2023-12-31
Phoenix,518,2022-08-01,2023-12-31


missing share per column:


pm25    0.46
pm10    0.46
o3      0.46
no2     0.46
uv      0.53
Name: % missing, dtype: float64